In [56]:
#user_input = input("Enter News Article: ")
title = "BREAKING: Secret Government Document Confirms Banned Energy Technology Silenced Forever!"
body = '''In a shocking turn of events that the mainstream media is completely refusing to cover, an anonymous high-level whistleblower has finally stepped forward with absolute proof of a massive cover-up. According to unverified documents leaked late last night, top-tier global scientists successfully engineered an entirely free, self-sustaining alternative energy source capable of powering the entire planet without a single ounce of fossil fuels or electricity grids.
Sources claim that the groundbreaking project was immediately seized by an alliance of shadowy corporate billionaires and corrupt politicians during a midnight raid on the laboratory. The brilliant inventors were reportedly paid millions to vanish forever, ensuring that the public remains trapped under the thumb of rising utility costs.
"They don't want you to know the truth," the unnamed whistleblower stated in an encrypted video transmission. "This technology would have liberated humanity, but the greed of the elite knows no bounds. Pass this along to everyone before it gets taken down!"
While official representatives from the Department of Energy have completely denied the existence of any such raid or technology, independent internet researchers note that the sudden removal of three minor web forums last Tuesday is absolute validation that a digital purge is actively underway. Do not let them silence the truth!'''

In [57]:
from dotenv import load_dotenv
import os

load_dotenv()
from langchain_groq  import ChatGroq

llm = ChatGroq(
    model = "qwen/qwen3.8-27b",
    temperature= 0.7
)

In [58]:
search_llm = ChatGroq(
    model= 'llama-3.1-8b-instant',
    temperature= 0.7
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

search_prompts = ChatPromptTemplate.from_messages([
    ("system", """You are an AI research query generator.
Analyze the user's article text and extract the core claim and entities.
Output ONLY ONE concise search query (under 8 words) for a search engine to fact-check this claim.
Do not output quotes, explanations, or extra commentary."""),
    ("human", "{search_input}")
])

In [60]:
from langchain_core.output_parsers import StrOutputParser
search_chain = search_prompts | search_llm |StrOutputParser()

In [ ]:
from duckduckgo_search import DDGS

'''def search_internet(user_input:str)-> str:
    try:
        # We are giving first 1500 characters to the llm groq api 
        search_query = search_chain.invoke({"search_input": user_input[:1500]})
        print(f"\n Generated search query to search query {search_query}....searching....")
    except Exception as e:
        # If groq api exhaustem free limits then we will give first 100 character to duckduckgo search
        print(f"\n[Query Generation Error]: {e}")
        search_query = f"{user_input[:80]} fact check"
    snippets = []
    try:
        with DDGS as ddgs:
            q = f"{search_query} fact check"
            results = ddgs.text(q, max_results=4)
            for r in results:
                snippets.append(
                f"Source: {r.get('href')}\nTitle: {r.get('title')}\nSnippet: {r.get('body')}"
                )
    except Exception as e:
            return f"404"
    return "\n\n---\n\n".join(snippets)'''

In [85]:
from ddgs import DDGS
def search_internet(user_input: str) -> str:
    # 1. Generate search query
    try:
        search_query = search_chain.invoke({"search_input": user_input[:1500]}).strip()
        print(f"\n[Generated Search Query]: {search_query} ...searching...")
    except Exception as e:
        print(f"[Query Chain Exception]: {e}")
        search_query = f"{user_input[:80]} fact check"
        print(f"[Fallback Query]: {search_query} ...searching...")

    # 2. DuckDuckGo search
    snippets = []
    try:
        with DDGS() as ddgs:  # Must have ()
            clean_query = search_query if "fact check" in search_query.lower() else f"{search_query} fact check"
            results = list(ddgs.text(clean_query, max_results=4))
            print(f"[DuckDuckGo Raw Results Count]: {len(results)}")
            
            for r in results:
                snippets.append(
                    f"Source: {r.get('href')}\nTitle: {r.get('title')}\nSnippet: {r.get('body')}"
                )
    except Exception as e:
        # DO NOT return "404" — print the real error
        print(f"[DDGS Exception]: {type(e).__name__} - {e}")
        return f"Search failed due to: {type(e).__name__} - {e}"

    if not snippets:
        return "No external fact checks found."

    return "\n\n---\n\n".join(snippets)

In [86]:
test_context = search_internet("Secret Government Document Confirms Banned Energy Technology")
print("\n--- FINAL RETRIEVED CONTEXT ---")
print(test_context)

[Query Chain Exception]: "Input to ChatPromptTemplate is missing variables {'context'}.  Expected: ['context', 'search_input'] Received: ['search_input']\nNote: if you intended {context} to be part of the string and not a variable, please escape it with double curly braces like: '{{context}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "
[Fallback Query]: Secret Government Document Confirms Banned Energy Technology fact check ...searching...
[DuckDuckGo Raw Results Count]: 4

--- FINAL RETRIEVED CONTEXT ---
Source: https://en.wikipedia.org/wiki/Free_energy_suppression_conspiracy_theory
Title: Free energy suppression conspiracy theory - Wikipedia
Snippet: Free energy suppression (or new energy suppression) is a conspiracy theory that technologically viable, pollution-free, no-cost energy sources are being suppressed by governments, corporations, or advocacy groups. [1][2] Devices allegedly suppressed include perpetual motion 

In [76]:
from langchain_core.prompts import MessagesPlaceholder
main_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a professional AI fact-checker. 
Analyze the user's article and compare it strictly against the verified external evidence provided in <context>.

Response format:
**Claims made**
- [Claim 1]
- [Claim 2]

**FACTS**
[✓ Fact Right] - State accurate claims (if any) with source citation.
[✗ Wrong Claim] - State false or unverified claims.
Explanation: 1 concise paragraph explaining why it is false using the context.

**Summary**
[1-2 sentences summarizing the verdict]

**Resource Citing**
- [List URLs from context used to verify]

<context>
{context}
</context>"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

In [77]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

def get_session_history(session_id:str) -> BaseChatMessageHistory:
    return SQLChatMessageHistory(
        session_id = session_id,
        connection = "sqlite:///chat_history.db"
    )


In [78]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (RunnablePassthrough.assign(
    context = lambda x: search_internet(x["input"])
    
) 
|main_prompt   
| llm)

In [79]:
chatbot = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key = "chat_history"
)

In [80]:
article_text = (
    "BREAKING: Secret Government Document Confirms Banned Energy Technology Silenced Forever! "
    "In a shocking turn of events that the mainstream media is completely refusing to cover, "
    "an anonymous high-level whistleblower has finally stepped forward with absolute proof of a massive cover-up. "
    "According to unverified documents leaked late last night, top-tier global scientists successfully "
    "engineered an entirely free, self-sustaining alternative energy source capable of powering the entire planet."
)

response = chatbot.invoke(
    {"input": article_text},
    config={"configurable": {"session_id": "session_001"}}
)

print(response)


[Query Generation Error]: "Input to ChatPromptTemplate is missing variables {'context'}.  Expected: ['context', 'search_input'] Received: ['search_input']\nNote: if you intended {context} to be part of the string and not a variable, please escape it with double curly braces like: '{{context}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "
content='**Claims made**\n- An anonymous high-level whistleblower provided proof of a massive government cover-up.\n- Top-tier global scientists successfully engineered a free, self-sustaining alternative energy source capable of powering the entire planet.\n- This technology is banned and mainstream media is refusing to cover it.\n\n**FACTS**\n[✗ Wrong Claim] - No verified evidence exists of a banned, free, self-sustaining energy source engineered by global scientists or a government cover-up regarding such a technology.\nExplanation: The provided context is "404," which indicates that no

In [81]:
response.content

'**Claims made**\n- An anonymous high-level whistleblower provided proof of a massive government cover-up.\n- Top-tier global scientists successfully engineered a free, self-sustaining alternative energy source capable of powering the entire planet.\n- This technology is banned and mainstream media is refusing to cover it.\n\n**FACTS**\n[✗ Wrong Claim] - No verified evidence exists of a banned, free, self-sustaining energy source engineered by global scientists or a government cover-up regarding such a technology.\nExplanation: The provided context is "404," which indicates that no external evidence, documents, or sources were found to verify any of the claims in the article. Furthermore, the text itself describes the documents as "unverified," and there is no established scientific or journalistic record of a proven, free, self-sustaining energy source that powers the entire planet being suppressed by governments.\n\n**Summary**\nThe article presents a fabricated conspiracy theory wit